In [1]:
print('hii')

hii


In [2]:
%pwd

'c:\\Users\\BIMLESH KUMAR SAH\\Desktop\\College Project 2\\College-chatbot\\research'

In [3]:
import os
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\BIMLESH KUMAR SAH\\Desktop\\College Project 2\\College-chatbot'

In [5]:
import json
from langchain.schema import Document

def load_json_chunks(json_path):
    documents = []

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for i, item in enumerate(data):
        text = item.get("text", "").strip()
        url = item.get("url", "")

        if text:  # skip empty text
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": url,
                        "chunk_id": i
                    }
                )
            )

    return documents

c:\Users\BIMLESH KUMAR SAH\Desktop\College Project 2\College-chatbot\collbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
documents = load_json_chunks("data/pims_knowledge_chunks.json")

In [7]:
documents

[Document(metadata={'source': 'https://www.pims.org.in/', 'chunk_id': 0}, page_content='Padmashree Institute of Management & Sciences Autonomous | Kommaghatta Apply Now 2026 Admissions 2026-27 Admission for PG/UG/Diploma Courses For Apply Online: https://admissions.pims.org.in/ Call 96321 71973 Contact 78994 08181 WhatsApp Click Here to pay\n        Online Examination Fee\n        payment Admissions Open Apply Now Overseas Students’ Admissions 100% Placement Assistance Campus Life 25+ Years of service 15+ Affiliations 60+ Courses 250+ Professors 3,000+ Students UG Courses Padmashree institutions have a variety of undergraduate courses that students can choose from based on their interests and passions. PG Courses Discover an array of specialized PG courses tailored to enhance your expertise and propel your career to new heights. Diploma Courses Diploma courses are shorter programs that assist students in specializing in a specific skill and entering the workforce quickly. After 10th, 1

In [8]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cpu'},   # change to 'cuda' if GPU
        encode_kwargs={'normalize_embeddings': True}
    )

    return embeddings

embedding = download_embeddings()

C:\Users\BIMLESH KUMAR SAH\AppData\Local\Temp\ipykernel_4580\151907746.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [9]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, multi_process=False, show_progress=False)

In [ ]:
vector = embedding.embed_query('hii')
vector

In [12]:
print( "Vector length:", len(vector))

Vector length: 384


In [13]:
from dotenv import load_dotenv
import os

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [14]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

In [15]:
pc

In [16]:
from pinecone import ServerlessSpec

index_name = "college-bot"

# Get existing indexes
existing_indexes = [index.name for index in pc.list_indexes()]

# Create index if it doesn't exist
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,  # change if using OpenAI (1536)
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# Connect to index
index = pc.Index(index_name)

In [18]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)

In [19]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

# ADD MORE DATA FOR FUTURE

In [29]:
from langchain.schema import Document

new_doc = Document(
    page_content=(
        "The admission process at Padmashree Institute of Management & Sciences (PIMS) involves the following steps: "
        "First, candidates must register themselves on the official admissions portal. "
        "Next, they need to verify their email ID and fill out the online application form. "
        "After completing the form, applicants are required to pay the application fee. "
        "Finally, they must submit the application form and can track their admission status through the portal. "
        "Applicants are advised to use the Query Management System (QMS) for any queries during the process."
    ),
    metadata={"source": "PIMS Admission Process"}
)

# Add to Pinecone
docsearch.add_documents([new_doc])

['c956f4c4-0b35-465e-8d66-e11e7cf4aee3']

In [31]:
new_doc = Document(
    page_content=(
        "Admission Process at PIMS:\n"
        "1. Register on the official admissions portal.\n"
        "2. Verify your email ID.\n"
        "3. Fill out the online application form.\n"
        "4. Pay the application fee.\n"
        "5. Submit the application.\n"
        "6. Track status via dashboard or QMS."
    ),
    metadata={"source": "PIMS Admission Process"}
)

In [32]:
index = pc.Index("college-bot")

print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 9605}},
 'total_vector_count': 9605,
 'vector_type': 'dense'}


In [ ]:
docsearch.add_documents(documents=documents)

In [21]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

retrieved_docs = retriever.invoke("What is location of padmashree?")

In [22]:
retrieved_docs

[Document(id='ddaba92c-3759-4188-a219-5d60a916fb0d', metadata={'chunk_id': 65.0, 'source': 'https://www.pims.org.in/page/contact-us'}, page_content='rg.in For admissions contact: 9632171973 / 78994 08181 / 6366-377377 / 6366-377388 Address: Padmashree Campus Sy. No. 149, Kommaghatta Sulikere Post, Kengeri Bengaluru – 560060 ADMISSION ENQUIRY 2026 Close modal ADMISSION ENQUIRY 2026 Close'),
 Document(id='bbd3e052-43c9-4eb9-a4ab-2512e41f7663', metadata={'chunk_id': 65.0, 'source': 'https://www.pims.org.in/page/contact-us'}, page_content='rg.in For admissions contact: 9632171973 / 78994 08181 / 6366-377377 / 6366-377388 Address: Padmashree Campus Sy. No. 149, Kommaghatta Sulikere Post, Kengeri Bengaluru – 560060 ADMISSION ENQUIRY 2026 Close modal ADMISSION ENQUIRY 2026 Close'),
 Document(id='27f73021-1cea-4e06-98fc-3057a9a57b36', metadata={'chunk_id': 4762.0, 'source': 'https://www.pims.org.in/storage/files/4/3.1.1/3.1.1 Projects sanction - DVV.pdf'}, page_content='t guides are hereby dir

In [23]:
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

chatModel = ChatOpenAI(
    model="gpt-4o",
    temperature=0.3
)

system_prompt = (
    "You are EVEREST, the official college assistant chatbot for Padmashree Institute of Management & Sciences (PIMS). "
    "Use only the retrieved context below to answer the user's question. "
    "If the answer cannot be found in the context, say clearly that you do not have that information in the provided knowledge base. "
    "Do not hallucinate, invent facts, or answer beyond the given context. "
    "Your responses should sound natural, intelligent, helpful, and clear like ChatGPT. "
    "You must automatically choose the best response format based on the user's intent and the nature of the answer. "
    "Use a concise direct answer for simple questions. "
    "Use paragraphs for explanations or descriptive answers. "
    "Use bullet points or numbered steps for lists, procedures, requirements, facilities, benefits, or comparisons. "
    "Use headings when the answer has multiple sections. "
    "If the user requests a specific format, always follow that format exactly. "
    "Keep answers well-structured, readable, and relevant. "
    "Prefer clarity over unnecessary length."
    "\n\n"
    "Retrieved context:\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({
    "input": "Tell me about hostel facilities at PIMS"
})

print(response["answer"])

I do not have specific information about the hostel facilities at Padmashree Institute of Management & Sciences (PIMS) in the provided context. For detailed information, I recommend contacting the institute directly or visiting their official website.


In [24]:
response = rag_chain.invoke({"input": "What courses are offered at PIMS?"})
print(response["answer"])


Padmashree Institute of Management & Sciences (PIMS) offers a variety of courses across different levels:

- **Undergraduate (UG) Courses**: A range of programs that students can choose based on their interests and passions, including a B.Sc. in Food Sciences & Nutrition.

- **Postgraduate (PG) Courses**: Specialized courses designed to enhance expertise and advance careers.

- **Diploma Courses**: Shorter programs that help students specialize in specific skills and enter the workforce quickly.

For more detailed information about specific courses, you may want to visit their official website or contact them directly.


In [ ]:
response = rag_chain.invoke({"input": "List the campus facilities."})
print(response["answer"])

The campus facilities at Padmashree Institute of Management & Sciences include:

- Telephone and internet connectivity across all departments, library, labs, computer lab, and office.
- Specified parking space for staff and students.
- Separate internet café for students and staff during off college hours.
- UPS and standby generator for power backup during load shedding and power cuts.
- Air-conditioned labs to maintain precision and avoid dust.
- Adequate number of bore wells for continuous water supply.
- Water purifiers installed for drinking water.
- 39 spacious classrooms, with 27 being ICT-enabled.
- 14 labs for practical classes.
- 9 research labs, including a central instrumentation room and animal culture rooms.
- 6 seminar halls with ICT facilities.
- Computer lab and language lab with modern accessories.
- Adequate staff room with toilet facilities.
- Wi-Fi enabled campus.
- Total of 105 computers available.


In [ ]:
response = rag_chain.invoke({"input": "Give me the contact number"})
print(response["answer"])

For admissions, you can contact the following numbers:

- 9632171973
- 7899408181
- 6366-377377
- 6366-377388


In [27]:
response = rag_chain.invoke({"input": "who are you?"})
print(response["answer"])

I am EVEREST, the official college assistant chatbot for Padmashree Institute of Management & Sciences (PIMS). I'm here to help answer your questions using the information available in the provided knowledge base. If you have any questions about the institute, feel free to ask!


In [30]:
response = rag_chain.invoke({"input": "What is the admission process? Answer in points."})
print(response["answer"])

I do not have specific information about the admission process for Padmashree Institute of Management & Sciences (PIMS) in the provided knowledge base. For detailed admission procedures, I recommend contacting the institute directly or visiting their official website.
